# Archive Exploration Starter

Audience: PILATES users who have a completed archived run and want a notebook-first way to inspect outcomes.

Prerequisites:
- a readable archive run directory
- a Consist DB under `.consist/`
- tabular outputs with schema metadata for grouped Ibis views

By the end you should be able to:
1. open an archive,
2. inspect the run catalog,
3. request a faceted Ibis table,
4. measure outcomes,
5. compare over time, over iterations, or across a parameter facet.


## 1) Configure Paths

Set `ARCHIVE_RUN_DIR` to a completed PILATES archive. `LOCAL_CACHE` is optional but useful on HPC because it copies the DB and selected artifacts to node-local storage as you use them.


In [ ]:
import os
from pathlib import Path

import ibis

from pilates_consist_analysis import delta, delta_change, difference, open_archive, rank

ARCHIVE_RUN_DIR = Path("/path/to/archive/run")
PROJECT_ROOT = Path("/Users/zaneedell/git/PILATES")
LOCAL_CACHE = Path("/scratch") / os.environ.get("USER", "user") / "pilates-analysis"
USE_LOCAL_CACHE = False


## 2) Open The Archive

`open_archive` is the main entry point. Use `local_cache=LOCAL_CACHE` when the archive is on slow persistent storage and the node has local scratch.


In [ ]:
archive = open_archive(
    ARCHIVE_RUN_DIR,
    project_root=PROJECT_ROOT,
    local_cache=LOCAL_CACHE if USE_LOCAL_CACHE else None,
)

display(archive.summary())
archive.issues()


## 3) Inspect Runs And Facets

Start with the run catalog. It tells you which scenario IDs, years, iterations, models, and seeds are available before you touch large output tables.


In [ ]:
runs = archive.runs()
print("Scenarios:", archive.scenarios())
print("Years:", archive.years())
print("Models:", archive.models())
display(runs.head(20))


## 4) Request A Faceted Table

The table name is `<model>.<logical_output>`. Requested facets become ordinary columns. Keep the result as an Ibis table until you need to display rows.


In [ ]:
trips = archive.table(
    "activitysim.trips",
    facets=["scenario_id", "year", "iteration", "pricing_policy", "trip_mode"],
    where={},
)

trips.limit(5).to_pandas()


## 5) Measure Outcomes

Measurements are normal Ibis aggregations grouped by the facets you care about. This example uses the checked-in ActivitySim trips schema and computes mode share from `trip_mode`.


In [ ]:
counts = archive.measure(
    trips,
    by=["scenario_id", "pricing_policy", "year", "iteration", "trip_mode"],
    measures={"trip_count": lambda table: table.count()},
)
total_window = ibis.window(
    group_by=[counts.scenario_id, counts.pricing_policy, counts.year, counts.iteration],
)
mode_shares = counts.mutate(
    total_trips=counts.trip_count.sum().over(total_window),
    mode_share=counts.trip_count / counts.trip_count.sum().over(total_window),
)

mode_shares.limit(20).to_pandas()


## 6) Compare Over Time, Iterations, Or Parameter Facets

These helpers do boring comparison math. They do not know what a scenario means; they only use columns.


In [ ]:
over_time = delta(
    mode_shares,
    value="mode_share",
    over="year",
    by=["scenario_id", "pricing_policy", "trip_mode"],
)

over_iterations = delta(
    mode_shares,
    value="mode_share",
    over="iteration",
    by=["scenario_id", "pricing_policy", "year", "trip_mode"],
)

iteration_acceleration = delta_change(
    mode_shares,
    value="mode_share",
    over="iteration",
    by=["scenario_id", "pricing_policy", "year", "trip_mode"],
)

parameter_difference = difference(
    mode_shares,
    value="mode_share",
    compare="pricing_policy",
    baseline="none",
    at={"year": 2030},
    by=["scenario_id", "iteration", "trip_mode"],
)

parameter_rank = rank(
    mode_shares,
    value="mode_share",
    by=["year", "iteration", "trip_mode"],
)


## 7) Display Small Results

Call `.to_pandas()` at the boundary. Keep previews small in teaching notebooks.


In [ ]:
display(over_time.limit(20).to_pandas())
display(parameter_difference.limit(20).to_pandas())
display(parameter_rank.limit(20).to_pandas())


## Exercise

Change the workflow to compare over `scenario_id` instead of `pricing_policy`.

Hint: request `scenario_id` as a facet, group by it, and call `difference(..., compare="scenario_id", baseline="baseline")`.


In [ ]:
# scenario_difference = difference(
#     mode_shares,
#     value="mode_share",
#     compare="scenario_id",
#     baseline="baseline",
#     at={"year": 2030},
#     by=["pricing_policy", "iteration", "trip_mode"],
# )
# display(scenario_difference.limit(20).to_pandas())


## Common Pitfall

If `archive.table(...)` fails with a missing schema or grouped-view error, the output was probably not profiled when it was logged. Use `archive.runs()` and `archive.tracker.find_artifacts_by_params(...)` to confirm the artifact family exists, then rerun the producer with schema profiling enabled for tabular outputs.
